<a href="https://colab.research.google.com/github/Marquittos11/nlp-autocompletado-LSTM/blob/main/notebooks/01_autocompletado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalación de librerías necesarias
!pip install PyPDF2
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.3 MB/s eta 0:00:00


In [2]:
!pip install PyPDF2

In [5]:
# --- Fase 1: Preprocesamiento del Texto ---
import PyPDF2
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences # IMPORTACIÓN CORREGIDA
import pickle

# 1. Configuración de rutas y extracción
ruta_archivo = "/content/documento.pdf"

def extraer_texto_pdf(ruta):
    texto_completo = ""
    with open(ruta, 'rb') as archivo:
        lector_pdf = PyPDF2.PdfReader(archivo)
        for pagina in lector_pdf.pages:
            texto_pagina = pagina.extract_text()
            if texto_pagina:
                texto_pagina = texto_pagina.replace('\n', ' ')
                texto_completo += texto_pagina + " "
    return texto_completo

def limpiar_texto(texto_crudo):
    texto = texto_crudo.lower()
    texto = re.sub(r'[^a-záéíóúñ\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

texto_crudo = extraer_texto_pdf(ruta_archivo)
texto = limpiar_texto(texto_crudo)

# 2. Tokenización y Vocabulario
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts([texto])
vocab_size = len(tokenizer.word_index) + 1
print(f"Tamaño del vocabulario: {vocab_size} palabras únicas.")

# 3. Generación de Secuencias de Ventana Fija
secuencia_completa = tokenizer.texts_to_sequences([texto])[0]
input_sequences = []
LONGITUD_VENTANA = 6 # 5 palabras de contexto + 1 a predecir

for i in range(LONGITUD_VENTANA, len(secuencia_completa)):
    n_gram_sequence = secuencia_completa[i - LONGITUD_VENTANA : i]
    input_sequences.append(n_gram_sequence)

# 4. Padding y Estructuración de Tensores
# CORRECCIÓN: Como los bloques ya son de 6, solo lo convertimos a matriz Numpy
input_sequences = np.array(input_sequences)

# 5. Separación de X e y
X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

# 6. Guardar Tokenizer
with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("¡Procesamiento completado!")
print(f"Forma de X (Entradas): {X.shape}")
print(f"Forma de Y (Etiquetas): {y.shape}")

Tamaño del vocabulario: 166 palabras únicas.
¡Procesamiento completado!
Forma de X (Entradas): (255, 5)
Forma de Y (Etiquetas): (255, 166)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# --- SEGUNDA FASE: LSTM y Entrenamiento ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# 1. DEFINIR LA ARQUITECTURA
modelo = Sequential()

# Capa 1: Embedding (Incrustación)
# Convierte identificadores numéricos en vectores densos que capturan la semántica.
# Cambiamos max_sequence_len-1 por 5 para que coincida con nuestra ventana fija.
#Al tener output_dim=100, las palabras tienen un mapa de significado mucho más rico.
modelo.add(Embedding(input_dim=vocab_size, output_dim=100, input_length=5))

# Capa 2: LSTM (Memoria a Corto y Largo Plazo)
# Orquesta los mecanismos de compuertas para gestionar la memoria secuencial.
# Esta celda utiliza el Carrusel de Error Constante (CEC) para evitar el desvanecimiento del gradiente.
#La cantidad de neuronas fijadas son 150 y se añade return_sequences = True porque pasa toda la secuencia temporal
# a la siguiente capa LSTM y no solo su resultado.
modelo.add(LSTM(units=150, return_sequences=True))

# Capa 3: Prevención de memorización
# Apaga aleatoriamente el 20% de las neuronas en cada paso. Obliga a la red a "aprender"
# patrones robustos en lugar de depender de neuronas específicas (evita el overfitting).
modelo.add(Dropout(0.2))

# Capa 4: LSTM
# Se establece una segunda capa para entender patrones gramaticales más complejos.
modelo.add(LSTM(units=100))

# Capa 5 : Capa Densa
# Proyecta los datos contra el léxico total usando Softmax para obtener probabilidades.
modelo.add(Dense(units=vocab_size, activation='softmax'))

# 2. COMPILAR EL MODELO
# Usamos categorical_crossentropy porque nuestras etiquetas 'y' están en formato One-Hot.
#Se parametrizo el learning_rate en 0.001 es un aprendizaje más lento pero evita que el modelo se desestabilice
# y salte la respuesta correcta.
modelo.compile(loss='categorical_crossentropy',
               optimizer=Adam(learning_rate=0.001),
               metrics=['accuracy'])

# Imprimimos un resumen de la arquitectura
print("--- RESUMEN DE LA ARQUITECTURA ---")
modelo.summary()

# 3. ENTRENAR EL MODELO (Con Parada Inteligente)
# MEJORA: EarlyStopping vigilará el entrenamiento. Si la red deja de mejorar durante 10 epochs seguidos,
# detendrá el entrenamiento automáticamente para no perder tiempo ni sobreajustar.
parada_temprana = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

print("\nIniciando el entrenamiento profundo...")
# Nota: Ahora usamos callbacks para inyectar nuestra "parada inteligente"
historial = modelo.fit(X, y, epochs=500, verbose=1, callbacks=[parada_temprana])

print("¡Entrenamiento finalizado con éxito!")

# Guardamos el modelo
modelo.save('modelo_autocompletado_profundo.h5')
print("¡El cerebro mejorado ha sido guardado como 'modelo_autocompletado_profundo.h5'!")

--- RESUMEN DE LA ARQUITECTURA ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Iniciando el entrenamiento profundo...
Epoch 1/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.0157 - loss: 5.1109
Epoch 2/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0667 - loss: 5.0968
Epoch 3/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0549 - loss: 5.0643
Epoch 4/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0510 - loss: 4.9394
Epoch 5/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0510 - loss: 4.7761
Epoch 6/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0510 - loss: 4.6856
Epoch 7/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0510 - loss: 4.6220
Epoch 8/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0510 - loss: 4.5439
Epoch 9/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0549 - loss: 4.4778
Epoch 10/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0549 - loss: 4.4057
Epoch 11/500
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0588 - loss: 4.3396
Epoch 12/500
8/8 ━━━━━━━━━━━━━━━━━

¡Entrenamiento finalizado con éxito!
¡El cerebro mejorado ha sido guardado como 'modelo_autocompletado_profundo.h5'!


In [7]:
# --- TERCERA FASE: Generador de Autocompletado ---
import numpy as np
from tensorflow.keras.utils import pad_sequences

def predecir_coherente(texto_semilla, n_palabras=4, temperatura=0.7, factor_penalizacion=1.5):
    """
    Genera texto utilizando Temperatura y Penalización de Repetición.
    """
    # Aquí guardaremos la memoria a cortísimo plazo de las palabras que vamos adivinando
    palabras_generadas = []

    for _ in range(n_palabras):
        # 1. Validación y Tokenización
        secuencias = tokenizer.texts_to_sequences([texto_semilla])

        # VALIDACIÓN CRÍTICA
        if len(secuencias[0]) == 0:
            print("\n⚠️ Error: El texto introducido no contiene palabras válidas en nuestro vocabulario.")
            return texto_semilla

        tokens = secuencias[0]

        # 2. Relleno (Padding) usando la variable dinámica de la Fase 1
        tokens = pad_sequences([tokens], maxlen=LONGITUD_VENTANA - 1, padding='pre')

        # 3. Predicción Pura
        probas = modelo.predict(tokens, verbose=0)[0]

        # Apagar la predicción de relleno (índice 0)
        probas[0] = 0.0

        # 4. PENALIZACIÓN DE REPETICIÓN (La cura para el "el el")
        # Si la palabra ya la generamos en esta frase, dividimos su probabilidad
        for id_gen in palabras_generadas:
            probas[id_gen] /= factor_penalizacion

        # 5. Aplicar Temperatura
        # Agregamos 1e-7 para evitar el logaritmo de cero (que da error matemático)
        probas = np.log(probas + 1e-7) / temperatura
        exp_probas = np.exp(probas)
        probas = exp_probas / np.sum(exp_probas) # Normalizamos a 100%

        # 6. Selección de palabra ganadora
        indice = np.random.choice(range(vocab_size), p=probas)

        # Guardamos el índice numérico en nuestro historial de repeticiones
        palabras_generadas.append(indice)

        # 7. Decodificación a texto humano
        palabra_final = ""
        for palabra, i in tokenizer.word_index.items():
            if i == indice:
                palabra_final = palabra
                break

        texto_semilla += " " + palabra_final

    return texto_semilla

# --- PRUEBA EN VIVO ---
# Prueba tu código modificado aquí. Si repite mucho, sube la temperatura (ej. 0.8)
# o sube el factor_penalizacion (ej. 2.0 o 3.0)
frase_prueba = "Francisco Cantuña fue contratado"
resultado = predecir_coherente(frase_prueba, n_palabras=5, temperatura=0.6, factor_penalizacion=2.0)

print("-" * 50)
print(f"Predicción final: {resultado}")
print("-" * 50)

Predicción: Francisco Cantuña fue contratado el el padres franciscanos
